# 02 — เสถียรภาพของฟีเจอร์ (Feature Stability)

ตรวจสอบว่าฟีเจอร์ Phase 2 ไม่พังที่ขอบ rollover และไม่มี look-ahead leakage

**หมายเหตุ (data-gated):** การรีวิว 5 ปีเต็มต้องใช้ `TVKIT_AUTH_TOKEN` จริงเพื่อดึงข้อมูลย้อนหลัง
(เหมือน backfill ของ Phase 1) โน้ตบุคนี้รันบนข้อมูลที่มีอยู่ใน `data/features/` ถ้ายังไม่มีให้รัน
`uv run python scripts/build_features.py` ก่อน

In [1]:
from __future__ import annotations

import polars as pl

from tfex_s50_multi_tf_swing.config.settings import get_settings
from tfex_s50_multi_tf_swing.features.models import FeatureConfig
from tfex_s50_multi_tf_swing.features.store import FeatureStore

settings = get_settings()
store = FeatureStore(settings.data_dir, FeatureConfig())

TIMEFRAME = "5m"
try:
    panel = store.read_panel(TIMEFRAME)
    print(f"loaded {TIMEFRAME} panel: {panel.shape}")
except Exception as exc:  # noqa: BLE001 - notebook guard
    panel = None
    print(f"no feature panel yet ({exc}); run scripts/build_features.py first")

loaded 5m panel: (20281, 29)


## สัดส่วน null ต่อฟีเจอร์ (lookback ตอนต้น series)

Null ที่ส่วนหัวของ series เป็นเรื่องปกติ (ยังมี lookback ไม่พอ) — เราต้องการให้ส่วนกลาง/ท้าย
ไม่มี null ผิดปกติ

In [2]:
if panel is not None:
    null_frac = panel.null_count() / panel.height
    print(null_frac)

shape: (1, 29)
┌──────┬───────────┬────────────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ time ┆ timeframe ┆ ema_slope_ ┆ ema_slope_ ┆ … ┆ rv_percent ┆ trend_pers ┆ range_com ┆ volume_ex │
│ ---  ┆ ---       ┆ 20         ┆ 50         ┆   ┆ ile        ┆ istence    ┆ pression  ┆ pansion   │
│ f64  ┆ f64       ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│      ┆           ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64        ┆ f64       ┆ f64       │
╞══════╪═══════════╪════════════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 0.0  ┆ 0.0       ┆ 0.005868   ┆ 0.007347   ┆ … ┆ 0.012968   ┆ 0.000986   ┆ 0.0       ┆ 0.000937  │
└──────┴───────────┴────────────┴────────────┴───┴────────────┴────────────┴───────────┴───────────┘


## การกระจายของฟีเจอร์ก่อน/หลัง rollover

เปรียบเทียบ distribution ของฟีเจอร์หลัก (เช่น `atr_ratio`, `ema_slope_20`, `dist_from_vwap`)
ในหน้าต่างก่อน/หลังสัปดาห์ rollover เพื่อยืนยันว่าฟีเจอร์ทำงานบน continuous series ที่ปรับฐานแล้ว
โดยไม่มี jump ที่ขอบสัญญา

In [3]:
if panel is not None:
    cols = [c for c in ("atr_ratio", "ema_slope_20", "dist_from_vwap") if c in panel.columns]
    if cols:
        print(panel.select(cols).describe())
    else:
        print("none of the expected feature columns are present in the panel")

shape: (9, 4)
┌────────────┬───────────┬──────────────┬────────────────┐
│ statistic  ┆ atr_ratio ┆ ema_slope_20 ┆ dist_from_vwap │
│ ---        ┆ ---       ┆ ---          ┆ ---            │
│ str        ┆ f64       ┆ f64          ┆ f64            │
╞════════════╪═══════════╪══════════════╪════════════════╡
│ count      ┆ 20182.0   ┆ 20162.0      ┆ 20182.0        │
│ null_count ┆ 99.0      ┆ 119.0        ┆ 99.0           │
│ mean       ┆ -0.017506 ┆ -0.002984    ┆ -0.012086      │
│ std        ┆ 1.088214  ┆ 1.257375     ┆ 1.214662       │
│ min        ┆ -3.15299  ┆ -3.81697     ┆ -3.849844      │
│ 25%        ┆ -0.82576  ┆ -0.901718    ┆ -0.972922      │
│ 50%        ┆ -0.23688  ┆ -0.06154     ┆ -0.012557      │
│ 75%        ┆ 0.670032  ┆ 0.893533     ┆ 0.926706       │
│ max        ┆ 6.981224  ┆ 3.781495     ┆ 3.706796       │
└────────────┴───────────┴──────────────┴────────────────┘
